# 03c · YOLO 검출과 ROI 수위 회귀

이 파일은 DCC 2026 작업 노트북의 **공개용 보존본**입니다. 아래 원래 셀과 텍스트 출력은 저장된 실험 기록이며, 이번 공개 작업에서 모델 학습·추론·평가를 다시 실행하지 않았습니다.

- **목적:** 예측 bbox의 ROI를 ConvNeXt-Tiny로 회귀하고, 미검출 시 full-frame 모델을 사용하는 후보를 만듭니다.
- **입력:** 03A shard와 bbox, 03B full-frame bundle; 모드에 맞는 선택 기록
- **출력:** YOLO/ROI checkpoint, predicted-ROI validation 점수와 candidate bundle
- **실제 저장된 실행 상태:** 현재 설정과 저장 bundle은 pilot 기록입니다. predicted-ROI validation RMSE=0.33143462721504496 m, GT-crop oracle=0.22844103774769947 m를 기록합니다. oracle은 배포 후보 점수와 구분해야 합니다. 이 파일만으로 03D full_promote가 요구하는 full ROI 실행 전체를 확인할 수 없습니다.

**실행 전:** 공개 저장소에는 원천 이미지·라벨 ZIP, private test 연결표·정답, 모델 가중치, 원래 GCS 산출물이 없습니다. `YOUR_GCP_PROJECT_ID`, `YOUR_GCS_BUCKET`과 필요한 데이터·권한을 직접 설정해야 하며, 원래 봉인 파일의 generation·SHA-256·config 검증도 충족해야 합니다. placeholder만 바꾼다고 과거 산출물이 재생성되거나 모든 gate가 통과하는 것은 아닙니다.

**공개본 변경:** 개인 프로젝트·버킷·계정·로컬 데이터 경로를 placeholder로 바꾸고 Colab/Drive/작성자/widget metadata를 제거했습니다. 코드 셀의 순서와 계산·무결성 검증 로직은 보존했습니다. 모든 내장 image/HTML/JavaScript/기타 binary 출력과 markdown 첨부 이미지를 제거하고, 숫자·표·오류를 포함한 텍스트 출력은 식별자를 가린 뒤 보존했습니다. 사진이 없는 것은 해당 셀을 실행하지 않았다는 뜻이 아닙니다. 원본 code cell의 `execution_count`는 모두 `null`이므로 이 값만으로 실행 여부를 판단하지 않습니다.

[GitHub 코드](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/03c_roi_regression.ipynb) · [Colab에서 열기](https://colab.research.google.com/github/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/03c_roi_regression.ipynb) · [실행 관계와 상태](README.md) · [원본 이름·SHA-256 대응표](../docs/notebook-source-map.csv)

---

아래는 비공개 식별자를 치환한 원래 노트북입니다. 원래 안내의 예정 단계·성공 기준은 위의 실제 저장 상태와 구분해 읽으십시오.


# DCC 2026 Mission 3 — 03C. YOLO26s + ConvNeXt-Tiny ROI 수위 회귀

이 노트북은 배포 가능한 두 단계 후보를 만듭니다.

1. YOLO26s가 원본 이미지 픽셀에서 `mokja` bbox를 예측
2. 고정 ConvNeXt-Tiny ROI evaluator가 crop과 파일명 첫 문자 유형으로 meter 수위를 회귀
3. 미검출이면 03B의 고정 full-frame 승자를 fallback으로 사용

GT bbox crop의 RMSE는 detector 손실을 분리하는 oracle 진단값입니다. 최종 후보 점수에는
predicted bbox만 사용합니다.


## clean GPU runtime

이 노트북은 Ultralytics를 설치하므로 03B를 실행했던 runtime에서 이어서 실행하지 마십시오.
pilot과 full도 각각 새 GPU runtime에서 첫 설치 셀부터 실행합니다. T4 기준 YOLO26s 960px
batch 4, ROI 384px batch 16을 시작값으로 사용합니다. OOM이면 batch 값을 낮추고 새 RUN_ID로
다시 시작하십시오. L4/A100은 full 실행의 권장 runtime입니다.


In [ ]:
%pip -q install "ultralytics==8.4.121" "google-cloud-storage>=2.19,<4" "pyarrow>=18,<24" "pandas>=2.2,<3"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.4/65.4 kB 7.2 MB/s eta 0:00:00


In [ ]:
import gc
import hashlib
import io
import json
import math
import os
import platform
import random
import shutil
import tarfile
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import ultralytics
from google.colab import auth
from PIL import Image
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from ultralytics import YOLO

if ultralytics.__version__ != "8.4.121":
    raise RuntimeError(f"Ultralytics version 불일치: {ultralytics.__version__}")

PROJECT_ID = "YOUR_GCP_PROJECT_ID"
BUCKET_NAME = "YOUR_GCS_BUCKET"
RUN_MODE = "pilot_candidate"  # "smoke", "pilot_candidate", "full_retrain"
PILOT_SELECTION_ID = "m3-selection-pilot-v1"
if RUN_MODE == "full_retrain":
    DATASET_VERSION = "m3ds-v1-full"
    DATA_PROFILE = "full"
    BASELINE_RUN_ID = "m3-fullframe-full-v1"
    RUN_ID = "m3-yolo26s-roi-full-v1"
    RUN_LEVEL = "full"
else:
    DATASET_VERSION = "m3ds-v1-pilot"
    DATA_PROFILE = "pilot"
    BASELINE_RUN_ID = "m3-fullframe-pilot-v1"
    RUN_ID = "m3-yolo26s-roi-smoke-v1" if RUN_MODE == "smoke" else "m3-yolo26s-roi-pilot-v1"
    RUN_LEVEL = "smoke" if RUN_MODE == "smoke" else "pilot"

ROI_IMAGE_SIZE = 384
ROI_BATCH_SIZE = 16
ROI_EPOCHS = 3 if RUN_LEVEL == "full" else (5 if RUN_LEVEL == "pilot" else 1)
ROI_LEARNING_RATE = 2e-4
CROP_PADDING = 0.15
TRAIN_JITTER = 0.04

YOLO_WEIGHTS = "yolo26s.pt" if RUN_LEVEL != "smoke" else "yolo26n.pt"
YOLO_IMGSZ = 960 if RUN_LEVEL != "smoke" else 640
YOLO_BATCH = 4
YOLO_EPOCHS = 4 if RUN_LEVEL == "full" else (8 if RUN_LEVEL == "pilot" else 1)
YOLO_FRACTION = 1.0 if RUN_LEVEL != "smoke" else 0.02
YOLO_CONFIDENCE = 0.05
YOLO_MAX_DET = 10
DETECTOR_FULL_MAX_ROWS = 60_000
SEED = 42
EXPECTED_FULL_VAL_N = 30_808

LOCAL_ROOT = Path("/content/dcc_mission3_03c")
LOCAL_SHARDS = LOCAL_ROOT / "shards" / DATASET_VERSION
LOCAL_IMAGES = LOCAL_ROOT / "images_all"
LOCAL_VAL_IMAGES = LOCAL_ROOT / "validation_images" / DATASET_VERSION
LOCAL_YOLO = LOCAL_ROOT / "yolo_dataset"
LOCAL_RUN = LOCAL_ROOT / "runs" / RUN_ID
LOCAL_META = LOCAL_ROOT / "meta"
for path in [LOCAL_SHARDS, LOCAL_IMAGES, LOCAL_YOLO, LOCAL_RUN, LOCAL_META]:
    path.mkdir(parents=True, exist_ok=True)

RUN_PREFIX = f"dcc/mission3/runs/{RUN_ID}"
DATASET_SEAL_OBJECT = f"dcc/mission3/datasets/{DATASET_VERSION}/dataset_seal.json"
BASELINE_BUNDLE_OBJECT = f"dcc/mission3/runs/{BASELINE_RUN_ID}/candidate_bundle.json"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("YOLO26+ROI 학습에는 GPU runtime이 필요합니다")
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", torch.cuda.get_device_properties(0).total_memory / 1024**3)
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)
print("ultralytics:", ultralytics.__version__)


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU: Tesla T4
VRAM GiB: 14.56317138671875
torch: 2.11.0+cu128 torchvision: 0.26.0+cu128
ultralytics: 8.4.121


In [ ]:
from google.api_core.exceptions import NotFound, PreconditionFailed
from google.cloud import storage

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def canonical_sha256(value):
    raw = json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def canonical_target_sha256(frame, target_column):
    """원본 float64와 DataLoader float32가 같은 수위를 같은 byte 계약으로 해시합니다."""
    required = {"stem", "type", target_column}
    missing = required - set(frame.columns)
    if missing:
        raise AssertionError(f"target hash 필수 열 누락: {sorted(missing)}")
    ordered = frame.sort_values("stem", kind="mergesort")
    records = [
        {
            "stem": str(getattr(row, "stem")),
            "type": str(getattr(row, "type")),
            "target_m_float32": float(np.float32(float(getattr(row, target_column)))),
        }
        for row in ordered.itertuples(index=False)
    ]
    return canonical_sha256(records)

def parse_gs_uri(uri):
    if not str(uri).startswith("gs://"):
        raise ValueError(f"GCS URI가 아닙니다: {uri}")
    bucket_name, object_name = str(uri)[5:].split("/", 1)
    return bucket_name, object_name

def download_object(bucket, object_name, local_path, generation=None):
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    blob = bucket.blob(object_name, generation=generation)
    kwargs = {"checksum": "auto"}
    if generation is not None:
        kwargs["if_generation_match"] = int(generation)
    blob.download_to_filename(str(local_path), **kwargs)
    return local_path

def upload_create_only(bucket, local_path, object_name, artifact_type):
    local_path = Path(local_path)
    file_sha = sha256_file(local_path)
    blob = bucket.blob(object_name)
    blob.metadata = {
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }
    try:
        blob.upload_from_filename(
            str(local_path), if_generation_match=0, checksum="auto"
        )
    except PreconditionFailed:
        blob.reload()
        remote_sha = (blob.metadata or {}).get("sha256")
        if remote_sha != file_sha or int(blob.size or -1) != local_path.stat().st_size:
            raise RuntimeError(
                f"create-only 충돌: gs://{bucket.name}/{object_name}"
            )
    blob.reload()
    if (blob.metadata or {}).get("sha256") != file_sha:
        raise RuntimeError(f"업로드 SHA metadata 검증 실패: {object_name}")
    return {
        "uri": f"gs://{bucket.name}/{object_name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }

def upload_json_create_only(bucket, payload, object_name, artifact_type):
    local_path = LOCAL_META / Path(object_name).name
    local_path.parent.mkdir(parents=True, exist_ok=True)
    local_path.write_text(
        json.dumps(payload, ensure_ascii=False, sort_keys=True, indent=2) + "\n",
        encoding="utf-8",
    )
    return upload_create_only(bucket, local_path, object_name, artifact_type)

def download_json(bucket, object_name, generation=None):
    blob = bucket.blob(object_name, generation=generation)
    kwargs = {}
    if generation is not None:
        kwargs["if_generation_match"] = int(generation)
    return json.loads(blob.download_as_bytes(**kwargs).decode("utf-8"))

def remote_descriptor(bucket, object_name, artifact_type=None):
    blob = bucket.blob(object_name)
    blob.reload()
    metadata = blob.metadata or {}
    file_sha = metadata.get("sha256")
    if not file_sha:
        raise RuntimeError(f"remote object에 sha256 metadata가 없습니다: {object_name}")
    return {
        "uri": f"gs://{bucket.name}/{blob.name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": artifact_type or metadata.get("artifact_type", "unknown"),
    }


In [ ]:
TYPE_ORDER = list("ABCEFGHS")
TYPE_TO_INDEX = {name: index for index, name in enumerate(TYPE_ORDER)}
TYPE_RANGES = {
    "A": (0.0, 15.0), "B": (0.0, 9.0),
    "C": (0.0, 15.0), "E": (0.0, 10.0),
    "F": (0.0, 4.0),  "G": (0.0, 4.0),
    "H": (0.0, 20.0), "S": (0.0, 20.0),
}

class Letterbox:
    def __init__(self, size, fill=(114, 114, 114)):
        self.size = int(size)
        self.fill = fill

    def __call__(self, image):
        image = image.convert("RGB")
        width, height = image.size
        scale = min(self.size / width, self.size / height)
        new_size = (
            max(1, int(round(width * scale))),
            max(1, int(round(height * scale))),
        )
        resized = image.resize(new_size, Image.Resampling.BILINEAR)
        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        offset = ((self.size - new_size[0]) // 2, (self.size - new_size[1]) // 2)
        canvas.paste(resized, offset)
        return canvas

def build_transform(size, training=False, augment=False):
    steps = [transforms.Lambda(Letterbox(size))]
    if training and augment:
        steps.extend([
            transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.06),
            transforms.RandomAffine(
                degrees=2.0, translate=(0.02, 0.02), scale=(0.96, 1.04),
                fill=(114, 114, 114),
            ),
        ])
    steps.extend([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ])
    return transforms.Compose(steps)

def build_backbone(name, pretrained):
    if name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = models.resnet18(weights=weights)
        feature_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
    elif name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        backbone = models.efficientnet_b0(weights=weights)
        feature_dim = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
    elif name == "convnext_tiny":
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        backbone = models.convnext_tiny(weights=weights)
        feature_dim = backbone.classifier[2].in_features
        backbone.classifier[2] = nn.Identity()
    else:
        raise ValueError(f"지원하지 않는 backbone: {name}")
    return backbone, int(feature_dim)

class TypeConditionalRegressor(nn.Module):
    def __init__(self, backbone_name, pretrained=True, dropout=0.15):
        super().__init__()
        self.backbone_name = backbone_name
        self.backbone, feature_dim = build_backbone(backbone_name, pretrained)
        self.type_embedding = nn.Embedding(len(TYPE_ORDER), 32)
        self.head = nn.Sequential(
            nn.Linear(feature_dim + 32, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )
        lows = [TYPE_RANGES[name][0] for name in TYPE_ORDER]
        highs = [TYPE_RANGES[name][1] for name in TYPE_ORDER]
        self.register_buffer("type_low", torch.tensor(lows, dtype=torch.float32))
        self.register_buffer("type_high", torch.tensor(highs, dtype=torch.float32))

    def forward(self, images, type_index):
        features = self.backbone(images)
        type_features = self.type_embedding(type_index)
        raw = self.head(torch.cat([features, type_features], dim=1)).squeeze(1)
        normalized = torch.sigmoid(raw)
        low = self.type_low[type_index]
        high = self.type_high[type_index]
        return low + normalized * (high - low)

def regression_loss(pred_m, target_m, type_index):
    lows = pred_m.new_tensor([TYPE_RANGES[name][0] for name in TYPE_ORDER])
    highs = pred_m.new_tensor([TYPE_RANGES[name][1] for name in TYPE_ORDER])
    low = lows[type_index]
    span = highs[type_index] - low
    pred_norm = (pred_m - low) / span
    target_norm = (target_m - low) / span
    robust = F.smooth_l1_loss(pred_norm, target_norm, beta=0.05)
    meter_term = torch.mean(((pred_m - target_m) / 20.0) ** 2)
    return robust + 0.20 * meter_term

def metric_payload(
    frame, truth_column="target_m", prediction_column="pred_m",
    require_all_types=True,
):
    required = {"type", truth_column, prediction_column}
    if required - set(frame.columns):
        raise AssertionError(f"metric 열 누락: {sorted(required - set(frame.columns))}")
    values = frame[[truth_column, prediction_column]].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise AssertionError("metric 입력에 non-finite 값이 있습니다")
    error = frame[prediction_column].to_numpy(float) - frame[truth_column].to_numpy(float)
    overall = float(np.sqrt(np.mean(error ** 2)))
    by_type = {}
    present_types = [
        type_name for type_name in TYPE_ORDER
        if frame["type"].eq(type_name).any()
    ]
    if require_all_types and present_types != TYPE_ORDER:
        missing_types = [name for name in TYPE_ORDER if name not in present_types]
        raise AssertionError(f"metric에 유형이 없습니다: {missing_types}")
    for type_name in present_types:
        part = frame.loc[frame["type"].eq(type_name)]
        part_error = part[prediction_column].to_numpy(float) - part[truth_column].to_numpy(float)
        by_type[type_name] = {
            "n": int(len(part)),
            "rmse_m": float(np.sqrt(np.mean(part_error ** 2))),
            "mae_m": float(np.mean(np.abs(part_error))),
            "bias_m": float(np.mean(part_error)),
        }
    weighted_mse = sum(
        item["n"] * item["rmse_m"] ** 2 for item in by_type.values()
    ) / sum(item["n"] for item in by_type.values())
    if not np.isclose(overall ** 2, weighted_mse, rtol=1e-10, atol=1e-12):
        raise AssertionError("전체 RMSE와 유형별 weighted MSE가 일치하지 않습니다")
    return {
        "overall_rmse_m": overall,
        "worst_type_rmse_m": max(item["rmse_m"] for item in by_type.values()),
        "by_type": by_type,
    }


In [ ]:
auth.authenticate_user()
STORAGE_CLIENT = storage.Client(project=PROJECT_ID)
BUCKET = STORAGE_CLIENT.bucket(BUCKET_NAME)
BUCKET.reload()
dataset_seal = download_json(BUCKET, DATASET_SEAL_OBJECT)
baseline_bundle = download_json(BUCKET, BASELINE_BUNDLE_OBJECT)
assert dataset_seal["dataset_version"] == DATASET_VERSION
assert baseline_bundle["dataset_version"] == DATASET_VERSION
assert baseline_bundle["eligible_for_selection"] is True
if baseline_bundle.get("training_scope", "pilot") != (
    "full" if RUN_MODE == "full_retrain" else "pilot"
):
    raise AssertionError("fallback bundle training scope 불일치")
assert baseline_bundle["validation_stem_sha256"] == dataset_seal["validation_stem_sha256"]
assert baseline_bundle["validation_target_sha256"] == dataset_seal["validation_target_sha256"]
if dataset_seal["counts"]["val"] != EXPECTED_FULL_VAL_N:
    raise AssertionError("YOLO candidate 비교에는 전체 validation이 필요합니다")

parent_selection = None
if RUN_MODE == "full_retrain":
    parent_selection = download_json(
        BUCKET,
        f"dcc/mission3/selection/{PILOT_SELECTION_ID}/model_selection_seal.json",
    )
    if parent_selection["status"] != "MODEL_SELECTION_FROZEN":
        raise AssertionError("pilot selection seal 상태 오류")
    if parent_selection["winner"]["kind"] != "yolo_roi":
        raise RuntimeError("pilot 승자가 YOLO+ROI일 때만 03C full 재학습을 실행합니다")
    if parent_selection["winner_model_id"] != "yolo26s-roi-convnext-tiny":
        raise AssertionError("동결한 YOLO architecture ID 불일치")
    if dataset_seal["data_profile"] != "full" or dataset_seal["counts"]["train"] < 200_000:
        raise AssertionError("full 재학습용 전체 train dataset이 아닙니다")

fallback_id = baseline_bundle["preliminary_winner"]
fallback_candidate = next(
    item for item in baseline_bundle["candidates"] if item["model_id"] == fallback_id
)
if RUN_MODE == "full_retrain":
    expected_fallback_backbone = parent_selection["winner"]["fallback_backbone"]
    if fallback_candidate["backbone"] != expected_fallback_backbone:
        raise AssertionError("full fallback architecture가 pilot 동결값과 다릅니다")
print("fixed fallback:", fallback_id)


fixed fallback: fullframe-convnext_tiny


## 1. canonical shard를 local image와 YOLO label로 물질화

두 모델은 같은 `mokja_boxes_xyxy_norm`에서 파생된 bbox를 사용합니다. JPEG는 재인코딩하지 않고
hard link로 YOLO 폴더에 연결합니다. `detector_trainable=True`와 유효 bbox가 있는 표본만 detector
학습·validation에 포함하며, downstream validation에는 30,808개 모두를 유지합니다.


In [ ]:
TRAIN_SHARDS = dataset_seal["shards"]["train"]
VAL_SHARDS = dataset_seal["shards"]["val"]
required_extract_bytes = sum(
    int(item["payload_bytes"]) for item in TRAIN_SHARDS + VAL_SHARDS
)
if shutil.disk_usage("/content").free < required_extract_bytes + 20 * 1024**3:
    raise RuntimeError(
        "train+val image 추출 공간이 부족합니다. High-RAM runtime 또는 더 큰 local disk가 필요합니다."
    )

for split_name in ["train", "val"]:
    (LOCAL_IMAGES / split_name).mkdir(parents=True, exist_ok=True)
    (LOCAL_YOLO / "images" / split_name).mkdir(parents=True, exist_ok=True)
    (LOCAL_YOLO / "labels" / split_name).mkdir(parents=True, exist_ok=True)

def write_bytes_once(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.stat().st_size != len(payload):
            raise RuntimeError(f"기존 파일 충돌: {path}")
        return
    with open(path, "xb") as stream:
        stream.write(payload)

def hardlink_once(source, target):
    if target.exists():
        if not os.path.samefile(source, target):
            raise RuntimeError(f"기존 hardlink 충돌: {target}")
        return
    os.link(source, target)

def box_to_yolo(box):
    x1, y1, x2, y2 = map(float, box)
    return ((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1)

def unpack_split(split_name, shard_descriptors):
    records = []
    for shard_index, descriptor in enumerate(shard_descriptors, start=1):
        _, object_name = parse_gs_uri(descriptor["uri"])
        shard_path = (
            LOCAL_SHARDS / split_name / descriptor["archive_key"]
            / Path(object_name).name
        )
        if not shard_path.exists():
            download_object(
                BUCKET, object_name, shard_path, generation=descriptor["generation"]
            )
        if shard_path.stat().st_size != int(descriptor["bytes"]):
            raise AssertionError("shard size 불일치")
        if sha256_file(shard_path) != descriptor["sha256"]:
            raise AssertionError("shard SHA 불일치")
        with tarfile.open(shard_path, "r:") as handle:
            members = {member.name: member for member in handle.getmembers()}
            keys = sorted({Path(name).stem for name in members})
            for key in keys:
                image_member = members.get(f"{key}.jpg")
                meta_member = members.get(f"{key}.json")
                if image_member is None or meta_member is None:
                    raise AssertionError(f"tar sample pair 누락: {key}")
                image_bytes = handle.extractfile(image_member).read()
                metadata = json.loads(handle.extractfile(meta_member).read().decode("utf-8"))
                if metadata["stem"] != key or metadata["split"] != split_name:
                    raise AssertionError("tar key/metadata 불일치")
                image_path = LOCAL_IMAGES / split_name / f"{key}.jpg"
                write_bytes_once(image_path, image_bytes)
                boxes = metadata.get("mokja_boxes_xyxy_norm", [])
                detector_ok = bool(metadata.get("detector_trainable")) and bool(boxes)
                if detector_ok:
                    yolo_image = LOCAL_YOLO / "images" / split_name / f"{key}.jpg"
                    yolo_label = LOCAL_YOLO / "labels" / split_name / f"{key}.txt"
                    hardlink_once(image_path, yolo_image)
                    lines = [
                        "0 " + " ".join(f"{value:.8f}" for value in box_to_yolo(box))
                        for box in boxes
                    ]
                    label_text = "\n".join(lines) + "\n"
                    if yolo_label.exists():
                        if yolo_label.read_text(encoding="utf-8") != label_text:
                            raise RuntimeError(f"기존 label 충돌: {yolo_label}")
                    else:
                        yolo_label.write_text(label_text, encoding="utf-8")
                records.append({
                    "stem": key,
                    "type": metadata["type"],
                    "target_m": float(metadata["target_m"]),
                    "image_path": str(image_path),
                    "boxes": boxes,
                    "detector_trainable": detector_ok,
                })
        shard_path.unlink(missing_ok=True)
        if shard_index % 10 == 0 or shard_index == len(shard_descriptors):
            print(split_name, "unpacked", shard_index, "/", len(shard_descriptors))
    frame = pd.DataFrame(records).sort_values("stem", kind="mergesort").reset_index(drop=True)
    assert frame["stem"].is_unique
    return frame

train_meta = unpack_split("train", TRAIN_SHARDS)
val_meta = unpack_split("val", VAL_SHARDS)
assert len(val_meta) == EXPECTED_FULL_VAL_N
assert canonical_sha256(sorted(val_meta["stem"].tolist())) == dataset_seal["validation_stem_sha256"]
observed_val_target_sha = canonical_target_sha256(val_meta, "target_m")
assert observed_val_target_sha == dataset_seal["validation_target_sha256"]
if RUN_LEVEL == "smoke":
    eval_meta = (
        val_meta.groupby("type", sort=False, group_keys=False)
        .head(8)
        .sort_values("stem", kind="mergesort")
        .reset_index(drop=True)
    )
    if set(eval_meta["type"]) != set(TYPE_ORDER):
        raise AssertionError("smoke validation에 8개 유형이 모두 필요합니다")
else:
    eval_meta = val_meta
EXPECTED_EVAL_N = len(eval_meta)

detector_candidates = train_meta.loc[train_meta["detector_trainable"]].copy()
if RUN_LEVEL == "full" and len(detector_candidates) > DETECTOR_FULL_MAX_ROWS:
    detector_candidates["level_decile"] = (
        detector_candidates.groupby("type", observed=True)["target_m"]
        .transform(lambda values: pd.qcut(values.rank(method="first"), 10, labels=False))
        .astype(int)
    )
    stratum_counts = detector_candidates.groupby(
        ["type", "level_decile"], observed=True
    ).size()
    exact_quotas = stratum_counts / stratum_counts.sum() * DETECTOR_FULL_MAX_ROWS
    quotas = np.floor(exact_quotas).astype(int)
    for stratum in (exact_quotas - quotas).sort_values(ascending=False).index[
        : int(DETECTOR_FULL_MAX_ROWS - quotas.sum())
    ]:
        quotas.loc[stratum] += 1
    selected_stems = set()
    for stratum, quota in quotas.items():
        type_name, decile = stratum
        part = detector_candidates.loc[
            detector_candidates["type"].eq(type_name)
            & detector_candidates["level_decile"].eq(decile)
        ].copy()
        part["stable_order"] = part["stem"].map(
            lambda stem: hashlib.sha256(
                f"{SEED}|detector-full|{stem}".encode("utf-8")
            ).hexdigest()
        )
        selected_stems.update(
            part.sort_values("stable_order", kind="mergesort")
            .head(int(quota))["stem"].astype(str)
        )
    if len(selected_stems) != DETECTOR_FULL_MAX_ROWS:
        raise AssertionError("detector full stratified subset 수량 오류")
    excluded = detector_candidates.loc[
        ~detector_candidates["stem"].isin(selected_stems), "stem"
    ]
    for stem in excluded.astype(str):
        (LOCAL_YOLO / "images" / "train" / f"{stem}.jpg").unlink(missing_ok=True)
        (LOCAL_YOLO / "labels" / "train" / f"{stem}.txt").unlink(missing_ok=True)
    detector_train_rows = len(selected_stems)
    detector_subset_policy = "type_x_within_type_level_decile_proportional_hashed_60000"
else:
    detector_train_rows = len(detector_candidates)
    detector_subset_policy = "all_detector_trainable_rows"

# Ultralytics는 val=False여도 마지막 epoch에 validation을 수행할 수 있습니다. smoke에서는
# detector용 val 폴더 자체를 유형당 8개로 줄여 마지막 검증도 짧게 유지합니다.
if RUN_LEVEL == "smoke":
    smoke_detector_val = (
        val_meta.loc[val_meta["detector_trainable"]]
        .groupby("type", sort=False, group_keys=False)
        .head(8)
    )
    if set(smoke_detector_val["type"]) != set(TYPE_ORDER):
        raise AssertionError("smoke detector validation에 8개 유형이 모두 필요합니다")
    allowed_val_stems = set(smoke_detector_val["stem"].astype(str))
    for image_path in (LOCAL_YOLO / "images" / "val").glob("*.jpg"):
        if image_path.stem not in allowed_val_stems:
            image_path.unlink(missing_ok=True)
            (LOCAL_YOLO / "labels" / "val" / f"{image_path.stem}.txt").unlink(
                missing_ok=True
            )
    print("smoke detector val rows:", len(allowed_val_stems))
print("detector train rows:", detector_train_rows, detector_subset_policy)
print("train detector rows:", int(train_meta["detector_trainable"].sum()), "/", len(train_meta))
print("val detector rows:", int(val_meta["detector_trainable"].sum()), "/", len(val_meta))


train unpacked 10 / 23
train unpacked 20 / 23
train unpacked 23 / 23
val unpacked 10 / 30
val unpacked 20 / 30
val unpacked 30 / 30
detector train rows: 23999 all_detector_trainable_rows
train detector rows: 23999 / 24000
val detector rows: 30807 / 30808


In [ ]:
yolo_yaml = LOCAL_YOLO / "data.yaml"
yolo_yaml.write_text(
    "\n".join([
        f"path: {LOCAL_YOLO}",
        "train: images/train",
        "val: images/val",
        "names:",
        "  0: mokja",
        "",
    ]),
    encoding="utf-8",
)
print(yolo_yaml.read_text(encoding="utf-8"))


path: /content/dcc_mission3_03c/yolo_dataset
train: images/train
val: images/val
names:
  0: mokja



## 2. 공통 ROI evaluator 학습

train에서는 GT bbox에 15% padding과 작은 위치 jitter를 적용합니다. validation oracle은 jitter 없는
GT bbox를 사용합니다. 복수 bbox 표본은 가장 큰 유효 bbox를 선택하며 이 규칙은 detector 후보
모두에 고정합니다.


In [ ]:
def largest_box(boxes):
    return max(
        boxes,
        key=lambda box: (float(box[2]) - float(box[0])) * (float(box[3]) - float(box[1])),
    )

def padded_crop(image, box, padding, jitter=0.0):
    width, height = image.size
    x1, y1, x2, y2 = map(float, box)
    box_w, box_h = x2 - x1, y2 - y1
    center_x, center_y = (x1 + x2) / 2, (y1 + y2) / 2
    if jitter > 0:
        center_x += random.uniform(-jitter, jitter) * box_w
        center_y += random.uniform(-jitter, jitter) * box_h
    half_w = box_w * (0.5 + padding)
    half_h = box_h * (0.5 + padding)
    px1 = int(max(0, math.floor((center_x - half_w) * width)))
    py1 = int(max(0, math.floor((center_y - half_h) * height)))
    px2 = int(min(width, math.ceil((center_x + half_w) * width)))
    py2 = int(min(height, math.ceil((center_y + half_h) * height)))
    if px2 <= px1 or py2 <= py1:
        raise ValueError("degenerate crop")
    return image.crop((px1, py1, px2, py2))

class RoiDataset(Dataset):
    def __init__(self, frame, training):
        self.frame = frame.loc[frame["boxes"].map(bool)].reset_index(drop=True)
        self.training = bool(training)
        self.transform = build_transform(
            ROI_IMAGE_SIZE, training=training, augment=False
        )

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row["image_path"]) as source:
            image = source.convert("RGB")
        crop = padded_crop(
            image,
            largest_box(row["boxes"]),
            CROP_PADDING,
            jitter=TRAIN_JITTER if self.training else 0.0,
        )
        return (
            self.transform(crop),
            torch.tensor(TYPE_TO_INDEX[row["type"]], dtype=torch.long),
            torch.tensor(float(row["target_m"]), dtype=torch.float32),
            row["stem"],
            row["type"],
        )

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)

def roi_loader(frame, training):
    return DataLoader(
        RoiDataset(frame, training),
        batch_size=ROI_BATCH_SIZE,
        shuffle=training,
        num_workers=2,
        pin_memory=True,
        persistent_workers=False,
        worker_init_fn=seed_worker,
        drop_last=False,
    )

def evaluate_roi(model, frame):
    model.eval()
    records = []
    with torch.inference_mode():
        for images, type_index, targets, stems, types in roi_loader(frame, False):
            images = images.to(DEVICE, non_blocking=True)
            type_index = type_index.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=True):
                pred = model(images, type_index)
            for stem, type_name, target, value in zip(
                stems, types, targets.numpy(), pred.float().cpu().numpy()
            ):
                records.append({
                    "stem": str(stem), "type": str(type_name),
                    "target_m": float(target), "pred_m": float(value),
                })
    output = pd.DataFrame(records).sort_values("stem", kind="mergesort")
    return output, metric_payload(
        output, require_all_types=True
    )

ROI_CONFIG = {
    "backbone": "convnext_tiny",
    "image_size": ROI_IMAGE_SIZE,
    "batch_size": ROI_BATCH_SIZE,
    "epochs": ROI_EPOCHS,
    "crop_padding": CROP_PADDING,
    "train_jitter": TRAIN_JITTER,
    "learning_rate": ROI_LEARNING_RATE,
    "seed": SEED,
    "dataset_version": DATASET_VERSION,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda,
}
ROI_CONFIG_SHA256 = canonical_sha256(ROI_CONFIG)


In [ ]:
roi_model = TypeConditionalRegressor("convnext_tiny", pretrained=True).to(DEVICE)
roi_optimizer = AdamW(roi_model.parameters(), lr=ROI_LEARNING_RATE, weight_decay=1e-4)
roi_scheduler = CosineAnnealingLR(roi_optimizer, T_max=max(1, ROI_EPOCHS))
roi_scaler = torch.cuda.amp.GradScaler(enabled=True)
best_roi_rmse = float("inf")
roi_history = []
roi_start_epoch = 0

roi_rows = []
for blob in STORAGE_CLIENT.list_blobs(
    BUCKET_NAME, prefix=f"{RUN_PREFIX}/roi/checkpoints/epoch-"
):
    try:
        roi_rows.append((int(Path(blob.name).stem.split("-")[-1]), blob))
    except ValueError:
        continue
if roi_rows:
    roi_start_epoch, resume_blob = sorted(roi_rows, key=lambda item: item[0])[-1]
    resume_blob.reload()
    if not resume_blob.name.startswith(f"{RUN_PREFIX}/roi/checkpoints/epoch-"):
        raise AssertionError("ROI resume checkpoint prefix 불일치")
    if (resume_blob.metadata or {}).get("artifact_type") != "m3_roi_regressor_checkpoint":
        raise AssertionError("ROI resume checkpoint artifact type 불일치")
    resume_local = LOCAL_RUN / f"roi-resume-{roi_start_epoch:03d}.pt"
    download_object(
        BUCKET, resume_blob.name, resume_local, generation=resume_blob.generation
    )
    resume_state = torch.load(resume_local, map_location="cpu", weights_only=False)
    if resume_state.get("schema_version") != "dcc-m3-roi-checkpoint-v1":
        raise AssertionError("ROI resume checkpoint schema 불일치")
    if resume_state["roi_config_sha256"] != ROI_CONFIG_SHA256:
        raise RuntimeError("ROI resume config 충돌")
    roi_model.load_state_dict(resume_state["model"])
    roi_optimizer.load_state_dict(resume_state["optimizer"])
    roi_scheduler.load_state_dict(resume_state["scheduler"])
    roi_scaler.load_state_dict(resume_state["scaler"])
    best_roi_rmse = float(resume_state["best_roi_rmse"])
    roi_history = list(resume_state["history"])
    rng_state = resume_state.get("rng_state")
    if rng_state:
        random.setstate(rng_state["python"])
        np.random.set_state(rng_state["numpy"])
        torch.set_rng_state(rng_state["torch"])
        torch.cuda.set_rng_state_all(rng_state["cuda"])
    del resume_state, rng_state
    gc.collect()
    print("ROI resume epoch:", roi_start_epoch)

for epoch in range(roi_start_epoch + 1, ROI_EPOCHS + 1):
    roi_model.train()
    total_loss = 0.0
    seen = 0
    started = time.perf_counter()
    for batch_index, (images, type_index, targets, _, _) in enumerate(
        roi_loader(train_meta, True)
    ):
        if RUN_LEVEL == "smoke" and batch_index >= 2:
            break
        images = images.to(DEVICE, non_blocking=True)
        type_index = type_index.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        roi_optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=True):
            pred = roi_model(images, type_index)
            loss = regression_loss(pred, targets, type_index)
        roi_scaler.scale(loss).backward()
        roi_scaler.step(roi_optimizer)
        roi_scaler.update()
        total_loss += float(loss.detach()) * len(targets)
        seen += len(targets)
    roi_scheduler.step()
    oracle_frame, oracle_metrics = evaluate_roi(
        roi_model, eval_meta
    )
    current_rmse = oracle_metrics["overall_rmse_m"]
    is_best = current_rmse < best_roi_rmse
    if is_best:
        best_roi_rmse = current_rmse
    roi_history.append({
        "epoch": epoch,
        "train_loss": total_loss / max(1, seen),
        "oracle_val_rmse_m": current_rmse,
        "seconds": time.perf_counter() - started,
        "is_best": is_best,
    })
    checkpoint = {
        "schema_version": "dcc-m3-roi-checkpoint-v1",
        "roi_config": ROI_CONFIG,
        "roi_config_sha256": ROI_CONFIG_SHA256,
        "epoch": epoch,
        "best_roi_rmse": best_roi_rmse,
        "history": roi_history,
        "model": roi_model.state_dict(),
        "optimizer": roi_optimizer.state_dict(),
        "scheduler": roi_scheduler.state_dict(),
        "scaler": roi_scaler.state_dict(),
        "rng_state": {
            "python": random.getstate(),
            "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all(),
        },
    }
    local_checkpoint = LOCAL_RUN / f"roi-epoch-{epoch:03d}.pt"
    torch.save(checkpoint, local_checkpoint)
    descriptor = upload_create_only(
        BUCKET,
        local_checkpoint,
        f"{RUN_PREFIX}/roi/checkpoints/epoch-{epoch:03d}.pt",
        "m3_roi_regressor_checkpoint",
    )
    del checkpoint
    gc.collect()
    print("ROI", roi_history[-1])

if not roi_history:
    raise RuntimeError("ROI best checkpoint가 없습니다")
best_roi_epoch = int(min(roi_history, key=lambda row: row["oracle_val_rmse_m"])["epoch"])
best_roi_descriptor = remote_descriptor(
    BUCKET,
    f"{RUN_PREFIX}/roi/checkpoints/epoch-{best_roi_epoch:03d}.pt",
    "m3_roi_regressor_checkpoint",
)
_, best_roi_object = parse_gs_uri(best_roi_descriptor["uri"])
best_roi_local = LOCAL_RUN / "roi-best.pt"
download_object(
    BUCKET, best_roi_object, best_roi_local,
    generation=best_roi_descriptor["generation"],
)
best_roi_checkpoint = torch.load(best_roi_local, map_location="cpu", weights_only=False)
if best_roi_checkpoint.get("schema_version") != "dcc-m3-roi-checkpoint-v1":
    raise AssertionError("ROI best checkpoint schema 불일치")
if best_roi_checkpoint.get("roi_config_sha256") != ROI_CONFIG_SHA256:
    raise AssertionError("ROI best checkpoint config 불일치")
roi_model.load_state_dict(best_roi_checkpoint["model"])
del best_roi_checkpoint
gc.collect()
roi_model.to(DEVICE).eval()
oracle_frame, oracle_metrics = evaluate_roi(
    roi_model, eval_meta
)
print("GT crop oracle (deployment 후보 점수 아님):", oracle_metrics)
# detector 학습에는 ROI optimizer/model이 필요하지 않습니다. best checkpoint를 local에
# 남긴 뒤 GPU에서 제거하고, downstream validation 직전에 다시 로드합니다.
del roi_model, roi_optimizer, roi_scheduler, roi_scaler
gc.collect()
torch.cuda.empty_cache()


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 177MB/s]
/tmp/ipykernel_1520/3845412785.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  roi_scaler = torch.cuda.amp.GradScaler(enabled=True)


ROI resume epoch: 5


/tmp/ipykernel_1520/762557956.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True):


GT crop oracle (deployment 후보 점수 아님): {'overall_rmse_m': 0.22844103774769947, 'worst_type_rmse_m': 0.3763537729215215, 'by_type': {'A': {'n': 10521, 'rmse_m': 0.2518364868522401, 'mae_m': 0.09997266519022649, 'bias_m': 0.022465040333828946}, 'B': {'n': 3726, 'rmse_m': 0.07499692827105657, 'mae_m': 0.05021194044262635, 'bias_m': 0.03711281213719424}, 'C': {'n': 9194, 'rmse_m': 0.23194969534470103, 'mae_m': 0.09696205727608535, 'bias_m': 0.048629039146082284}, 'E': {'n': 2350, 'rmse_m': 0.17998265814659956, 'mae_m': 0.09717231094203096, 'bias_m': 0.027990372859417124}, 'F': {'n': 879, 'rmse_m': 0.13909862724002575, 'mae_m': 0.039600528234168805, 'bias_m': 0.011960082959735488}, 'G': {'n': 1098, 'rmse_m': 0.08452757403373812, 'mae_m': 0.037083220585013565, 'bias_m': 0.00037470044659786535}, 'H': {'n': 1031, 'rmse_m': 0.19553321388435443, 'mae_m': 0.10385942239419103, 'bias_m': 0.05755178623125463}, 'S': {'n': 2008, 'rmse_m': 0.3763537729215215, 'mae_m': 0.0967319206115734, 'bias_m': 0.064

## 3. YOLO26s `mokja` detector 학습

controlled arm에서는 flip, random crop 계열, mosaic, mixup, copy-paste와 색상 변형을 끕니다.
`on_model_save` callback이 매 checkpoint를 GCS에 create-only로 백업합니다.


In [ ]:
YOLO_CONFIG = {
    "weights": YOLO_WEIGHTS,
    "imgsz": YOLO_IMGSZ,
    "batch": YOLO_BATCH,
    "epochs": YOLO_EPOCHS,
    "fraction": YOLO_FRACTION,
    "max_det": YOLO_MAX_DET,
    "detector_train_rows": detector_train_rows,
    "detector_subset_policy": detector_subset_policy,
    "seed": SEED,
    "dataset_version": DATASET_VERSION,
    "ultralytics": ultralytics.__version__,
    "deterministic": True,
    "amp": True,
    "cache": False,
    "patience": 0,
    "save_period": 1,
    "augmentation": {
        "degrees": 0.0, "translate": 0.0, "scale": 0.0,
        "shear": 0.0, "perspective": 0.0,
        "hsv_h": 0.0, "hsv_s": 0.0, "hsv_v": 0.0,
        "mosaic": 0.0, "mixup": 0.0, "cutmix": 0.0,
        "copy_paste": 0.0, "fliplr": 0.0, "flipud": 0.0,
        "multi_scale": 0.0,
    },
}
YOLO_CONFIG_SHA256 = canonical_sha256(YOLO_CONFIG)
upload_json_create_only(
    BUCKET,
    {"yolo_config": YOLO_CONFIG, "yolo_config_sha256": YOLO_CONFIG_SHA256},
    f"{RUN_PREFIX}/detector/train_config.json",
    "m3_yolo26_train_config",
)

yolo_checkpoint_descriptors = []
def backup_yolo_checkpoint(trainer):
    actual_batch = int(trainer.args.batch)
    if actual_batch != int(YOLO_BATCH):
        raise RuntimeError(
            "Ultralytics가 OOM 뒤 batch를 자동 축소했습니다: "
            f"requested={YOLO_BATCH}, actual={actual_batch}. "
            "YOLO_BATCH를 실제 값으로 낮추고 새 RUN_ID로 다시 시작하십시오."
        )
    last_path = Path(trainer.last)
    if not last_path.exists():
        return
    epoch_number = int(trainer.epoch) + 1
    descriptor = upload_create_only(
        BUCKET,
        last_path,
        f"{RUN_PREFIX}/detector/checkpoints/epoch-{epoch_number:03d}.pt",
        "m3_yolo26_checkpoint",
    )
    yolo_checkpoint_descriptors.append(descriptor)
    best_path = Path(trainer.best)
    if best_path.exists():
        upload_create_only(
            BUCKET,
            best_path,
            f"{RUN_PREFIX}/detector/best_history/through-epoch-{epoch_number:03d}.pt",
            "m3_yolo26_best_checkpoint_history",
        )
    print("GCS detector backup:", descriptor["uri"])

# verified-skip 경로에서도 cleanup 변수가 항상 정의되도록 초기화합니다.
yolo_model = None
final_best_object = f"{RUN_PREFIX}/detector/best.pt"
final_best_blob = BUCKET.blob(final_best_object)
yolo_best_local = LOCAL_RUN / "yolo-best.pt"
if final_best_blob.exists():
    yolo_best_descriptor = remote_descriptor(
        BUCKET, final_best_object, "m3_yolo26_best_checkpoint"
    )
    final_best_blob.reload()
    if (final_best_blob.metadata or {}).get("artifact_type") != "m3_yolo26_best_checkpoint":
        raise AssertionError("YOLO best artifact type 불일치")
    download_object(
        BUCKET, final_best_object, yolo_best_local,
        generation=yolo_best_descriptor["generation"],
    )
    if sha256_file(yolo_best_local) != yolo_best_descriptor["sha256"]:
        raise AssertionError("기존 YOLO best checkpoint SHA 불일치")
    print("YOLO already complete -> verified skip")
else:
    checkpoint_rows = []
    for blob in STORAGE_CLIENT.list_blobs(
        BUCKET_NAME, prefix=f"{RUN_PREFIX}/detector/checkpoints/epoch-"
    ):
        try:
            checkpoint_rows.append((int(Path(blob.name).stem.split("-")[-1]), blob))
        except ValueError:
            continue
    if checkpoint_rows:
        resume_epoch, resume_blob = sorted(checkpoint_rows, key=lambda item: item[0])[-1]
        resume_blob.reload()
        if not resume_blob.name.startswith(f"{RUN_PREFIX}/detector/checkpoints/epoch-"):
            raise AssertionError("YOLO resume checkpoint prefix 불일치")
        if (resume_blob.metadata or {}).get("artifact_type") != "m3_yolo26_checkpoint":
            raise AssertionError("YOLO resume checkpoint artifact type 불일치")
        if resume_epoch >= YOLO_EPOCHS:
            history_rows = list(STORAGE_CLIENT.list_blobs(
                BUCKET_NAME,
                prefix=f"{RUN_PREFIX}/detector/best_history/through-epoch-",
            ))
            if not history_rows:
                raise RuntimeError("완료 epoch은 있으나 YOLO best history가 없습니다")
            latest_history = sorted(history_rows, key=lambda blob: blob.name)[-1]
            latest_history.reload()
            if (
                (latest_history.metadata or {}).get("artifact_type")
                != "m3_yolo26_best_checkpoint_history"
            ):
                raise AssertionError("YOLO best history artifact type 불일치")
            download_object(
                BUCKET, latest_history.name, yolo_best_local,
                generation=latest_history.generation,
            )
            yolo_model = None
            print("YOLO epochs already complete -> final best publish only")
        else:
            resume_local = LOCAL_RUN / f"yolo-resume-{resume_epoch:03d}.pt"
            download_object(
                BUCKET, resume_blob.name, resume_local, generation=resume_blob.generation
            )
            yolo_model = YOLO(str(resume_local))
            yolo_model.add_callback("on_model_save", backup_yolo_checkpoint)
            print("YOLO resume epoch:", resume_epoch)
            yolo_model.train(resume=True)
    else:
        yolo_model = YOLO(YOLO_WEIGHTS)
        yolo_model.add_callback("on_model_save", backup_yolo_checkpoint)
        yolo_model.train(
            data=str(yolo_yaml),
            imgsz=YOLO_IMGSZ,
            epochs=YOLO_EPOCHS,
            batch=YOLO_BATCH,
            device=0,
            workers=2,
            seed=SEED,
            deterministic=True,
            amp=True,
            cache=False,
            patience=0,
            save=True,
            save_period=1,
            project=str(LOCAL_RUN / "ultralytics"),
            name="yolo26-mokja",
            exist_ok=True,
            fraction=YOLO_FRACTION,
            degrees=0.0,
            translate=0.0,
            scale=0.0,
            shear=0.0,
            perspective=0.0,
            hsv_h=0.0,
            hsv_s=0.0,
            hsv_v=0.0,
            mosaic=0.0,
            mixup=0.0,
            cutmix=0.0,
            copy_paste=0.0,
            fliplr=0.0,
            flipud=0.0,
            multi_scale=0.0,
            plots=True,
            val=RUN_LEVEL != "smoke",
            verbose=True,
        )
    trainer_best = (
        Path(yolo_model.trainer.best) if yolo_model is not None else yolo_best_local
    )
    if trainer_best.exists():
        yolo_best_local = trainer_best
    else:
        history_rows = list(STORAGE_CLIENT.list_blobs(
            BUCKET_NAME, prefix=f"{RUN_PREFIX}/detector/best_history/through-epoch-"
        ))
        if not history_rows:
            raise RuntimeError("YOLO best checkpoint history가 없습니다")
        latest_history = sorted(history_rows, key=lambda blob: blob.name)[-1]
        latest_history.reload()
        if (
            (latest_history.metadata or {}).get("artifact_type")
            != "m3_yolo26_best_checkpoint_history"
        ):
            raise AssertionError("YOLO best history artifact type 불일치")
        download_object(
            BUCKET, latest_history.name, yolo_best_local,
            generation=latest_history.generation,
        )
    yolo_best_descriptor = upload_create_only(
        BUCKET,
        yolo_best_local,
        final_best_object,
        "m3_yolo26_best_checkpoint",
    )
completed_checkpoint_rows = []
for blob in STORAGE_CLIENT.list_blobs(
    BUCKET_NAME, prefix=f"{RUN_PREFIX}/detector/checkpoints/epoch-"
):
    try:
        completed_checkpoint_rows.append(
            (int(Path(blob.name).stem.split("-")[-1]), blob)
        )
    except ValueError:
        continue
if not completed_checkpoint_rows:
    raise RuntimeError("YOLO last checkpoint가 없습니다")
last_epoch, last_blob = sorted(completed_checkpoint_rows, key=lambda item: item[0])[-1]
yolo_last_descriptor = remote_descriptor(
    BUCKET, last_blob.name, "m3_yolo26_checkpoint"
)
print("YOLO best:", yolo_best_descriptor["uri"])
print("YOLO last:", yolo_last_descriptor["uri"], "epoch", last_epoch)
# 학습 trainer가 model/optimizer tensor를 계속 잡지 않도록 downstream validation 전에 해제합니다.
del yolo_model
gc.collect()
torch.cuda.empty_cache()


YOLO already complete -> verified skip
YOLO best: gs://YOUR_GCS_BUCKET/dcc/mission3/runs/m3-yolo26s-roi-pilot-v1/detector/best.pt
YOLO last: gs://YOUR_GCS_BUCKET/dcc/mission3/runs/m3-yolo26s-roi-pilot-v1/detector/checkpoints/epoch-008.pt epoch 8


## 4. predicted crop + 동일 ROI evaluator + 고정 fallback으로 전체 validation 평가

detection confidence는 crop 결합 가중치에만 사용합니다. 복수 crop은 confidence 가중 중앙값으로
합칩니다. 미검출이면 03B에서 미리 고정한 full-frame 모델을 사용합니다.


In [ ]:
def descriptor_download(descriptor, local_name, expected_prefix, allowed_artifact_types):
    bucket_name, object_name = parse_gs_uri(descriptor["uri"])
    if bucket_name != BUCKET_NAME or not object_name.startswith(expected_prefix):
        raise AssertionError("checkpoint bucket/prefix 불일치")
    if descriptor.get("artifact_type") not in set(allowed_artifact_types):
        raise AssertionError("checkpoint descriptor artifact type 불일치")
    pinned_blob = BUCKET.blob(object_name, generation=int(descriptor["generation"]))
    pinned_blob.reload()
    if (pinned_blob.metadata or {}).get("artifact_type") not in set(allowed_artifact_types):
        raise AssertionError("checkpoint remote artifact type 불일치")
    local = LOCAL_RUN / local_name
    download_object(BUCKET, object_name, local, generation=descriptor["generation"])
    if local.stat().st_size != int(descriptor["bytes"]):
        raise AssertionError("checkpoint byte 불일치")
    if sha256_file(local) != descriptor["sha256"]:
        raise AssertionError("checkpoint SHA 불일치")
    return local

fallback_local = descriptor_download(
    fallback_candidate["checkpoint"],
    "fallback-fullframe.pt",
    f"dcc/mission3/runs/{BASELINE_RUN_ID}/",
    {"m3_regressor_checkpoint"},
)
fallback_checkpoint = torch.load(fallback_local, map_location="cpu", weights_only=False)
if fallback_checkpoint.get("schema_version") != "dcc-m3-regressor-checkpoint-v1":
    raise AssertionError("fallback checkpoint schema 불일치")
if fallback_checkpoint.get("run_config_sha256") != baseline_bundle["run_config_sha256"]:
    raise AssertionError("fallback checkpoint run config 불일치")
fallback_backbone = fallback_candidate["backbone"]
fallback_model = TypeConditionalRegressor(fallback_backbone, pretrained=False)
fallback_model.load_state_dict(fallback_checkpoint["model"])
del fallback_checkpoint
gc.collect()
fallback_model.to(DEVICE).eval()
fallback_size = int(baseline_bundle["run_config"]["image_size"])
fallback_transform = build_transform(fallback_size, False, False)

class FullFrameDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row["image_path"]) as source:
            image = source.convert("RGB")
        return (
            fallback_transform(image),
            torch.tensor(TYPE_TO_INDEX[row["type"]], dtype=torch.long),
            row["stem"], row["type"], float(row["target_m"]),
        )

fallback_records = []
with torch.inference_mode():
    for images, type_index, stems, types, targets in DataLoader(
        FullFrameDataset(eval_meta), batch_size=32, shuffle=False,
        num_workers=0, pin_memory=False,
    ):
        images = images.to(DEVICE, non_blocking=True)
        type_index = type_index.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=True):
            pred = fallback_model(images, type_index)
        for stem, type_name, target, value in zip(
            stems, types, np.asarray(targets), pred.float().cpu().numpy()
        ):
            fallback_records.append({
                "stem": str(stem), "type": str(type_name),
                "target_m": float(target), "fallback_pred_m": float(value),
            })
fallback_frame = pd.DataFrame(fallback_records).set_index("stem")
assert len(fallback_frame) == EXPECTED_EVAL_N
del fallback_records, fallback_model
gc.collect()
torch.cuda.empty_cache()


/tmp/ipykernel_1520/639586338.py:62: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True):


In [ ]:
best_roi_checkpoint_for_eval = torch.load(
    best_roi_local, map_location="cpu", weights_only=False
)
if best_roi_checkpoint_for_eval.get("schema_version") != "dcc-m3-roi-checkpoint-v1":
    raise AssertionError("ROI evaluation checkpoint schema 불일치")
if best_roi_checkpoint_for_eval.get("roi_config_sha256") != ROI_CONFIG_SHA256:
    raise AssertionError("ROI evaluation checkpoint config 불일치")
roi_model = TypeConditionalRegressor("convnext_tiny", pretrained=False)
roi_model.load_state_dict(best_roi_checkpoint_for_eval["model"])
del best_roi_checkpoint_for_eval
gc.collect()
roi_model.to(DEVICE).eval()
roi_transform = build_transform(ROI_IMAGE_SIZE, False, False)

# Ultralytics에 30,808개 path list를 넘기면 list source가 in-memory image 입력으로
# 변환될 수 있습니다. validation 이미지만 hard-link한 폴더를 만들고 directory source로
# 전달하여 LoadImagesAndVideos가 한 장씩 읽도록 강제합니다.
if LOCAL_VAL_IMAGES.exists():
    shutil.rmtree(LOCAL_VAL_IMAGES)
LOCAL_VAL_IMAGES.mkdir(parents=True, exist_ok=False)
local_val_source_root = (LOCAL_IMAGES / "val").resolve()
expected_val_names = set()
for row in eval_meta.itertuples(index=False):
    source_path = Path(row.image_path).resolve()
    if source_path.parent != local_val_source_root:
        raise AssertionError(f"validation image source 경로 이탈: {source_path}")
    target_path = LOCAL_VAL_IMAGES / source_path.name
    if target_path.name in expected_val_names:
        raise AssertionError(f"validation image 파일명 중복: {target_path.name}")
    os.link(source_path, target_path)
    expected_val_names.add(target_path.name)
observed_val_names = {path.name for path in LOCAL_VAL_IMAGES.glob("*.jpg")}
if observed_val_names != expected_val_names or len(observed_val_names) != EXPECTED_EVAL_N:
    raise AssertionError("validation hard-link directory 완전성 실패")
print("lazy validation image directory:", len(observed_val_names))

def weighted_median(values_and_weights):
    ordered = sorted(values_and_weights, key=lambda pair: pair[0])
    total = sum(weight for _, weight in ordered)
    running = 0.0
    for value, weight in ordered:
        running += weight
        if running >= total / 2:
            return float(value)
    return float(ordered[-1][0])

def evaluate_detector_checkpoint(detector_path):
    detector_model = YOLO(str(detector_path))
    detector_checkpoint_payload = getattr(detector_model, "ckpt", None)
    if isinstance(detector_checkpoint_payload, dict):
        # Inference에는 optimizer/AMP scaler 상태가 필요하지 않습니다.
        detector_checkpoint_payload["optimizer"] = None
        detector_checkpoint_payload["scaler"] = None
    del detector_checkpoint_payload
    gc.collect()
    crop_inputs, crop_type_indices, crop_keys, crop_confidences = [], [], [], []
    crop_predictions = defaultdict(list)
    detection_summary = {}

    def flush_roi_buffer():
        if not crop_inputs:
            return
        images = torch.stack(crop_inputs).to(DEVICE)
        type_index = torch.tensor(crop_type_indices, dtype=torch.long, device=DEVICE)
        with torch.inference_mode(), torch.cuda.amp.autocast(enabled=True):
            values = roi_model(images, type_index).float().cpu().numpy()
        for key, confidence, value in zip(crop_keys, crop_confidences, values):
            crop_predictions[key].append((float(value), float(confidence)))
        crop_inputs.clear(); crop_type_indices.clear()
        crop_keys.clear(); crop_confidences.clear()

    prediction_stream = detector_model.predict(
        source=str(LOCAL_VAL_IMAGES),
        imgsz=YOLO_IMGSZ,
        batch=YOLO_BATCH,
        conf=YOLO_CONFIDENCE,
        max_det=YOLO_MAX_DET,
        device=0,
        stream=True,
        save=False,
        verbose=False,
    )
    for result_index, result in enumerate(prediction_stream, start=1):
        stem = Path(result.path).stem
        type_name = stem[0]
        with Image.open(result.path) as source:
            image = source.convert("RGB")
        boxes = (
            result.boxes.xyxy.detach().cpu().numpy()
            if result.boxes is not None else np.empty((0, 4))
        )
        confidences = (
            result.boxes.conf.detach().cpu().numpy()
            if result.boxes is not None else np.empty(0)
        )
        detection_summary[stem] = {
            "n_candidates": int(len(boxes)),
            "detector_conf": float(confidences.max()) if len(confidences) else float("nan"),
        }
        width, height = image.size
        for xyxy, confidence in zip(boxes, confidences):
            x1, y1, x2, y2 = map(float, xyxy)
            crop = padded_crop(
                image, [x1 / width, y1 / height, x2 / width, y2 / height],
                CROP_PADDING, jitter=0.0,
            )
            crop_inputs.append(roi_transform(crop))
            crop_type_indices.append(TYPE_TO_INDEX[type_name])
            crop_keys.append(stem)
            crop_confidences.append(max(float(confidence), 1e-6))
            if len(crop_inputs) >= 64:
                flush_roi_buffer()
        if result_index % 2_000 == 0:
            print("detector inference:", result_index, "/", EXPECTED_EVAL_N)
    flush_roi_buffer()
    if set(detection_summary) != set(eval_meta["stem"]):
        raise AssertionError("detector validation sample 완전성 실패")

    records = []
    for row in eval_meta.itertuples():
        crops = crop_predictions.get(row.stem, [])
        fallback_used = not crops
        pred_m = (
            float(fallback_frame.loc[row.stem, "fallback_pred_m"])
            if fallback_used else weighted_median(crops)
        )
        summary = detection_summary[row.stem]
        records.append({
            "stem": row.stem, "type": row.type,
            "target_m": float(row.target_m), "pred_m": pred_m,
            "detector_conf": summary["detector_conf"],
            "n_candidates": summary["n_candidates"],
            "fallback_used": fallback_used,
        })
    frame = pd.DataFrame(records).sort_values("stem", kind="mergesort")
    if len(frame) != EXPECTED_EVAL_N or not frame["stem"].is_unique:
        raise AssertionError("detector checkpoint validation 행 계약 실패")
    metrics = metric_payload(frame)
    del detector_model, prediction_stream, result, image, boxes, confidences
    gc.collect()
    torch.cuda.empty_cache()
    return frame, metrics

detector_descriptors = {
    item["sha256"]: item for item in [yolo_best_descriptor, yolo_last_descriptor]
}
detector_evaluations = []
detector_frames = {}
for descriptor in detector_descriptors.values():
    local_detector = descriptor_download(
        descriptor,
        f"detector-eval-{descriptor['sha256'][:12]}.pt",
        f"{RUN_PREFIX}/detector/",
        {"m3_yolo26_best_checkpoint", "m3_yolo26_checkpoint"},
    )
    frame, metrics = evaluate_detector_checkpoint(local_detector)
    detector_frames[descriptor["sha256"]] = frame
    detector_evaluations.append({"checkpoint": descriptor, "metrics": metrics})
    print(descriptor["uri"], metrics["overall_rmse_m"])
detector_evaluations.sort(key=lambda item: (
    item["metrics"]["overall_rmse_m"],
    item["metrics"]["worst_type_rmse_m"],
    item["checkpoint"]["sha256"],
))
selected_detector_evaluation = detector_evaluations[0]
selected_yolo_descriptor = selected_detector_evaluation["checkpoint"]
final_frame = detector_frames[selected_yolo_descriptor["sha256"]]
advanced_metrics = selected_detector_evaluation["metrics"]
print("YOLO26s + ROI validation metrics:")
print(json.dumps(advanced_metrics, ensure_ascii=False, indent=2))
print("selected detector checkpoint:", selected_yolo_descriptor["uri"])
print("fallback rate:", final_frame["fallback_used"].mean())


lazy validation image directory: 30808


/tmp/ipykernel_1520/2396580043.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.inference_mode(), torch.cuda.amp.autocast(enabled=True):


detector inference: 2000 / 30808
detector inference: 4000 / 30808
detector inference: 6000 / 30808
detector inference: 8000 / 30808
detector inference: 10000 / 30808
detector inference: 12000 / 30808
detector inference: 14000 / 30808
detector inference: 16000 / 30808
detector inference: 18000 / 30808
detector inference: 20000 / 30808
detector inference: 22000 / 30808
detector inference: 24000 / 30808
detector inference: 26000 / 30808
detector inference: 28000 / 30808
detector inference: 30000 / 30808
gs://YOUR_GCS_BUCKET/dcc/mission3/runs/m3-yolo26s-roi-pilot-v1/detector/best.pt 0.33143462721504496


/tmp/ipykernel_1520/2396580043.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.inference_mode(), torch.cuda.amp.autocast(enabled=True):


detector inference: 2000 / 30808
detector inference: 4000 / 30808
detector inference: 6000 / 30808
detector inference: 8000 / 30808
detector inference: 10000 / 30808
detector inference: 12000 / 30808
detector inference: 14000 / 30808
detector inference: 16000 / 30808
detector inference: 18000 / 30808
detector inference: 20000 / 30808
detector inference: 22000 / 30808
detector inference: 24000 / 30808
detector inference: 26000 / 30808
detector inference: 28000 / 30808
detector inference: 30000 / 30808
gs://YOUR_GCS_BUCKET/dcc/mission3/runs/m3-yolo26s-roi-pilot-v1/detector/checkpoints/epoch-008.pt 0.33143462721504496
YOLO26s + ROI validation metrics:
{
  "overall_rmse_m": 0.33143462721504496,
  "worst_type_rmse_m": 0.7097833916097381,
  "by_type": {
    "A": {
      "n": 10522,
      "rmse_m": 0.25028284381720173,
      "mae_m": 0.09909899446407304,
      "bias_m": 0.02205364848696854
    },
    "B": {
      "n": 3726,
      "rmse_m": 0.7097833916097381,
      "mae_m": 0.1682325427658891

## 5. 후보 bundle 게시

최종 후보에는 YOLO, ROI evaluator, full-frame fallback 세 checkpoint가 모두 포함됩니다.
validation prediction은 전체 30,808행이어야 selection에 참여할 수 있습니다.


In [ ]:
prediction_path = LOCAL_META / "validation_predictions.parquet"
final_frame.to_parquet(prediction_path, index=False, compression="zstd")
prediction_descriptor = upload_create_only(
    BUCKET,
    prediction_path,
    f"{RUN_PREFIX}/validation_predictions.parquet",
    "m3_validation_predictions",
)
diagnostics_path = LOCAL_META / "validation_diagnostics.parquet"
final_frame[[
    "stem", "detector_conf", "n_candidates", "fallback_used"
]].to_parquet(diagnostics_path, index=False, compression="zstd")
diagnostics_descriptor = upload_create_only(
    BUCKET,
    diagnostics_path,
    f"{RUN_PREFIX}/validation_diagnostics.parquet",
    "m3_validation_diagnostics",
)

observed_val_sha = canonical_sha256(sorted(final_frame["stem"].tolist()))
observed_target_sha = canonical_target_sha256(final_frame, "target_m")
if RUN_LEVEL != "smoke":
    assert observed_val_sha == dataset_seal["validation_stem_sha256"]
    assert observed_target_sha == dataset_seal["validation_target_sha256"]
candidate = {
    "model_id": "yolo26s-roi-convnext-tiny",
    "kind": "yolo_roi",
    "architecture_fingerprint": canonical_sha256({
        "kind": "yolo_roi",
        "detector": "yolo26s",
        "detector_imgsz": YOLO_IMGSZ,
        "roi_backbone": "convnext_tiny",
        "roi_image_size": ROI_IMAGE_SIZE,
        "crop_padding": CROP_PADDING,
        "detector_confidence": YOLO_CONFIDENCE,
        "max_detections": YOLO_MAX_DET,
        "multi_crop_fusion": "confidence_weighted_median",
        "fallback_backbone": fallback_backbone,
    }),
    "metrics": advanced_metrics,
    "validation_rows": int(len(final_frame)),
    "validation_stem_sha256": observed_val_sha,
    "validation_target_sha256": observed_target_sha,
    "prediction": prediction_descriptor,
    "diagnostics": diagnostics_descriptor,
    "checkpoint_bundle": {
        "detector": selected_yolo_descriptor,
        "roi_regressor": best_roi_descriptor,
        "fallback": fallback_candidate["checkpoint"],
    },
    "detector_config": YOLO_CONFIG,
    "detector_config_sha256": YOLO_CONFIG_SHA256,
    "detector_checkpoint_selection": {
        "rule": "minimum downstream validation meter RMSE, then worst-type RMSE",
        "evaluated": detector_evaluations,
        "selected_sha256": selected_yolo_descriptor["sha256"],
    },
    "roi_config": ROI_CONFIG,
    "roi_config_sha256": ROI_CONFIG_SHA256,
    "fallback_model_id": fallback_id,
    "fallback_source_run_id": baseline_bundle["run_id"],
    "fallback_backbone": fallback_backbone,
    "fallback_image_size": fallback_size,
    "inference_rule": {
        "detector_confidence": YOLO_CONFIDENCE,
        "max_detections": YOLO_MAX_DET,
        "crop_padding": CROP_PADDING,
        "multi_crop_fusion": "confidence_weighted_median",
        "fallback": fallback_id,
    },
    "oracle_diagnostic": {
        "eligible_as_deployment_candidate": False,
        "description": "GT largest mokja bbox crop + same ROI evaluator",
        "metrics": oracle_metrics,
        "rows": int(len(oracle_frame)),
    },
}
bundle = {
    "status": "CANDIDATE_COMPLETE",
    "schema_version": "dcc-m3-candidate-bundle-v1",
    "run_id": RUN_ID,
    "run_level": RUN_LEVEL,
    "eligible_for_selection": RUN_MODE in {"pilot_candidate", "full_retrain"},
    "training_scope": "full" if RUN_MODE == "full_retrain" else "pilot",
    "parent_selection_id": PILOT_SELECTION_ID if RUN_MODE == "full_retrain" else None,
    "dataset_version": DATASET_VERSION,
    "dataset_config_sha256": dataset_seal["config_sha256"],
    "validation_rows": int(len(final_frame)),
    "validation_stem_sha256": observed_val_sha,
    "validation_target_sha256": observed_target_sha,
    "candidate_count": 1,
    "candidates": [candidate],
    "package_versions": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "ultralytics": ultralytics.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
    },
}
bundle_descriptor = upload_json_create_only(
    BUCKET,
    bundle,
    f"{RUN_PREFIX}/candidate_bundle.json",
    "m3_candidate_bundle",
)
print("YOLO26 ROI CANDIDATE COMPLETE")
print("bundle:", bundle_descriptor["uri"])


YOLO26 ROI CANDIDATE COMPLETE
bundle: gs://YOUR_GCS_BUCKET/dcc/mission3/runs/m3-yolo26s-roi-pilot-v1/candidate_bundle.json


## 다음 단계

`DCC_Colab_03D_Mission3_Validation_선택봉인.ipynb`에서 같은 validation stem을 사용한 후보를
비교하고 승자를 create-only seal로 고정합니다.
